# Molmo2 — Multi-Image Pointing (Exploración)

Este notebook prueba la inferencia **por objeto completo**: se le pasan N vistas de un mismo objeto en un solo prompt y el modelo devuelve puntos de simetría para cada imagen.

### Estructura esperada del input
```
<renders_root>/<symmetry_type>/<object_id>/<image_size>/<illumination>/
    IND_00_AZ_..._ROT_000.png
    ...
    metadata_all.json
```

### Dependencias
```bash
pip install transformers==4.57.1 torch pillow einops torchvision accelerate molmo_utils
```

## 0. Configuración

In [ ]:
from pathlib import Path

# ── Ajusta estas rutas ────────────────────────────────────────────────────────
RENDERS_ROOT  = Path("../data/renders")   # raíz de los renders
SYMMETRY_TYPE = "axis_sym"                # "axis_sym" | "plane_sym"
OBJECT_ID     = "plane_example"           # nombre de la carpeta del objeto
IMAGE_SIZE    = 224                       # 224 | 448 | 1024
ILLUMINATION  = "flat"                    # "flat" | "darker" | "brighter"

# Grupos de vistas a probar (primeros N índices × 4 rotaciones)
# Empieza con grupos pequeños para entender el comportamiento del modelo
VIEW_GROUPS_TO_TEST = [6, 14, 26]

MODEL_ID = "allenai/Molmo2-8B"
# ─────────────────────────────────────────────────────────────────────────────

render_dir = RENDERS_ROOT / SYMMETRY_TYPE / OBJECT_ID / str(IMAGE_SIZE) / ILLUMINATION
print(f"Render dir: {render_dir}")
print(f"Existe: {render_dir.exists()}")

## 1. Cargar modelo y procesador

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

print(f"Cargando modelo: {MODEL_ID}")

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    dtype="auto",
    device_map="auto",
    padding_side="left",   # requerido para generación correcta
)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    dtype="auto",
    device_map="auto",
)
model.eval()

print(f"Dispositivo: {model.device}")
print("Modelo listo.")

## 2. Utilidades: cargar imágenes y parsear coordenadas

In [ ]:
import json
import re
from PIL import Image

# ── Regex oficial de Molmo2 para parsear puntos de imagen ─────────────────────
# Formato en el output: <points coords="R ID X Y ID X Y ...">
# donde R = radio (descartado), luego tripletas (ID, X, Y) en escala 0–1000
COORD_REGEX  = re.compile(r'<(?:points|point).*?coords="([^"]+)"')
POINTS_REGEX = re.compile(r'([0-9]+)\s+([0-9]{3,4})\s+([0-9]{3,4})')


def load_metadata(render_dir: Path) -> list[dict]:
    """Carga metadata_all.json desde la carpeta de renders."""
    meta_path = render_dir / "metadata_all.json"
    with open(meta_path, encoding="utf-8") as f:
        return json.load(f)


def get_entries_for_group(metadata: list[dict], n_views: int) -> list[dict]:
    """Retorna entradas cuyos índice de viewpoint < n_views."""
    return [m for m in metadata if m["index"] < n_views]


def load_images(entries: list[dict], render_dir: Path) -> list[Image.Image]:
    """Carga las imágenes PIL para las entradas dadas."""
    images = []
    for entry in entries:
        img_path = render_dir / entry["filename"]
        images.append(Image.open(img_path).convert("RGB"))
    return images


def extract_image_points(raw_output: str) -> list[dict]:
    """
    Parsea el output de Molmo2 para imagen(es) estáticas.
    
    Formato esperado: <points coords="R  ID X Y  ID X Y  ...">
    - R: radio (primer token, descartado)
    - Tripletas: (obj_id, x, y) en escala 0–1000
    
    Retorna lista de dicts: [{"obj_id": int, "x": float, "y": float}, ...]
    """
    points = []
    for coord_match in COORD_REGEX.finditer(raw_output):
        coord_str = coord_match.group(1)
        # Saltar el primer número (radio)
        numbers = coord_str.split()
        if len(numbers) < 4:
            continue
        coord_str_no_radius = " ".join(numbers[1:])
        for pt in POINTS_REGEX.finditer(coord_str_no_radius):
            points.append({
                "obj_id": int(pt.group(1)),
                "x": float(pt.group(2)),
                "y": float(pt.group(3)),
            })
    return points


print("Utilidades cargadas.")

## 3. Función de inferencia multi-imagen

In [ ]:
PROMPT = "Find the points where the main axis of symmetry intersects the edges of the shape."


def run_multiimage_inference(
    images: list[Image.Image],
    prompt: str = PROMPT,
    max_new_tokens: int = 500,
) -> dict:
    """
    Inferencia Molmo2 con N imágenes en un solo prompt.
    
    El modelo recibe todas las vistas del objeto juntas y devuelve
    coordenadas de puntos de simetría para el conjunto.
    
    Args:
        images:         Lista de imágenes PIL (vistas del mismo objeto).
        prompt:         Texto del prompt.
        max_new_tokens: Tokens máximos a generar.
    
    Returns:
        {
            "raw_output": str,        # texto completo generado
            "points":     list[dict], # [{obj_id, x, y}, ...] en coords 0–1000
            "n_images":   int,        # número de imágenes enviadas
        }
    """
    # Construir el content: texto primero, luego todas las imágenes
    # (el modelo fue entrenado con este orden para multi-image pointing)
    content = [{"type": "text", "text": prompt}]
    for img in images:
        content.append({"type": "image", "image": img})

    messages = [{"role": "user", "content": content}]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)

    raw_output = processor.tokenizer.decode(
        output_ids[0, inputs["input_ids"].size(1):],
        skip_special_tokens=True,
    )

    points = extract_image_points(raw_output)

    return {
        "raw_output": raw_output,
        "points": points,
        "n_images": len(images),
    }


print("Función de inferencia lista.")

## 4. Función de visualización

In [ ]:
import matplotlib.pyplot as plt
import math


def visualize_multiimage_results(
    images: list[Image.Image],
    entries: list[dict],
    points: list[dict],
    title: str = "",
    save_path: Path | None = None,
) -> None:
    """
    Muestra todas las imágenes en una grilla con los puntos superpuestos.

    NOTA: En modo multi-imagen el modelo devuelve puntos con obj_id
    que indican a qué imagen corresponde cada punto (1-based).
    Si el modelo no distingue por imagen, se dibujan en todas.

    Args:
        images:    Imágenes PIL en el mismo orden enviado al modelo.
        entries:   Metadatos correspondientes (para el título de cada imagen).
        points:    Lista de dicts {obj_id, x, y} en escala 0–1000.
        title:     Título del plot.
        save_path: Si se especifica, guarda la figura en disco.
    """
    n = len(images)
    cols = min(n, 6)
    rows = math.ceil(n / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3 + 0.5))
    axes = [axes] if n == 1 else axes.flatten()

    for i, (img, entry) in enumerate(zip(images, entries)):
        ax = axes[i]
        w, h = img.size
        ax.imshow(img)

        # Dibuja puntos cuyo obj_id coincide con este índice (1-based)
        # o todos los puntos si obj_id == 0 (el modelo no distingue)
        img_points = [
            p for p in points
            if p["obj_id"] == 0 or p["obj_id"] == (i + 1)
        ]

        for pt in img_points:
            # Escalar de 0–1000 a píxeles; Y invertido (origen Molmo = abajo)
            px = pt["x"] / 1000 * w
            py = h - pt["y"] / 1000 * h
            ax.scatter(px, py, s=80, c="red", edgecolors="white",
                       linewidths=1.5, zorder=10)
            ax.text(px + 4, py + 4, str(pt["obj_id"]),
                    color="white", fontsize=7, fontweight="bold", zorder=11)

        az  = entry.get("azimuth", "?")
        el  = entry.get("elevation", "?")
        rot = entry.get("rotation_deg", "?")
        ax.set_title(f"AZ{az} EL{el} R{rot}", fontsize=7)
        ax.axis("off")

    # Ocultar ejes sobrantes
    for j in range(n, len(axes)):
        axes[j].axis("off")

    n_pts = len(points)
    fig.suptitle(
        f"{title}  |  {n} imágenes  |  {n_pts} punto(s) detectado(s)",
        fontsize=10, y=1.01
    )
    plt.tight_layout()

    if save_path:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, bbox_inches="tight", dpi=100)
        print(f"Guardado en: {save_path}")

    plt.show()


print("Función de visualización lista.")

## 5. Experimento: inferencia por grupo de vistas

Para cada tamaño de grupo definido en `VIEW_GROUPS_TO_TEST`:
- Carga las primeras N×4 imágenes del objeto
- Las envía todas en un único prompt
- Muestra el output crudo y la visualización

In [ ]:
# Cargar metadata una sola vez
metadata = load_metadata(render_dir)
print(f"Total de imágenes disponibles: {len(metadata)}")
print(f"Ejemplo de entrada: {metadata[0]['filename']}")

In [ ]:
# Directorio de salida para visualizaciones
output_dir = Path("outputs") / OBJECT_ID / ILLUMINATION
output_dir.mkdir(parents=True, exist_ok=True)

results = {}  # guarda resultados por grupo para comparación posterior

for n_views in VIEW_GROUPS_TO_TEST:
    print("\n" + "=" * 60)
    print(f"Grupo: {n_views} vistas  →  {n_views * 4} imágenes")
    print("=" * 60)

    entries = get_entries_for_group(metadata, n_views)
    images  = load_images(entries, render_dir)

    print(f"Imágenes cargadas: {len(images)}")
    print("Ejecutando inferencia...")

    result = run_multiimage_inference(images)
    results[n_views] = result

    print(f"\n--- Output crudo del modelo ---")
    print(result["raw_output"])
    print(f"\n--- Puntos extraídos ({len(result['points'])}) ---")
    for p in result["points"]:
        print(f"  obj_id={p['obj_id']}  x={p['x']:.1f}  y={p['y']:.1f}")

    # Visualización
    visualize_multiimage_results(
        images  = images,
        entries = entries,
        points  = result["points"],
        title   = f"{OBJECT_ID} | {n_views} views | {ILLUMINATION}",
        save_path = output_dir / f"group_{n_views:03d}_views.png",
    )

## 6. Comparación de outputs por grupo

In [ ]:
print("\n=== Resumen comparativo ===")
print(f"{'Grupo':<10} {'Imágenes':<12} {'Puntos detectados':<20} {'¿Output vacío?'}")
print("-" * 55)

for n_views, result in results.items():
    n_imgs   = result["n_images"]
    n_points = len(result["points"])
    empty    = "SÍ" if not result["raw_output"].strip() else "no"
    print(f"{n_views:<10} {n_imgs:<12} {n_points:<20} {empty}")

## 7. Guardar resultados como JSON

In [ ]:
import json

for n_views, result in results.items():
    entries = get_entries_for_group(metadata, n_views)

    payload = {
        "object_id":     OBJECT_ID,
        "symmetry_type": SYMMETRY_TYPE,
        "image_size":    IMAGE_SIZE,
        "illumination":  ILLUMINATION,
        "n_views":       n_views,
        "n_images_sent": result["n_images"],
        "prompt":        PROMPT,
        "raw_output":    result["raw_output"],
        "points":        result["points"],
        "images_sent": [
            {
                "filename":    e["filename"],
                "index":       e["index"],
                "azimuth":     e["azimuth"],
                "elevation":   e["elevation"],
                "rotation_deg": e["rotation_deg"],
                "eye":         e["eye"],
                "R":           e["R"],
                "T":           e["T"],
            }
            for e in entries
        ],
    }

    json_path = output_dir / f"group_{n_views:03d}_views.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

    print(f"Guardado: {json_path}")

## 8. (Opcional) Comparar con inferencia imagen a imagen

Para el mismo grupo de vistas, corre la inferencia individualmente y compara cuántos puntos se detectan vs. el modo multi-imagen.

In [ ]:
# Elige un grupo para la comparación
N_VIEWS_COMPARE = VIEW_GROUPS_TO_TEST[0]  # el grupo más pequeño por velocidad

entries_cmp = get_entries_for_group(metadata, N_VIEWS_COMPARE)
images_cmp  = load_images(entries_cmp, render_dir)

print(f"Comparando modo imagen-a-imagen vs multi-imagen para grupo {N_VIEWS_COMPARE}...\n")

individual_points = []
for i, (img, entry) in enumerate(zip(images_cmp, entries_cmp)):
    result_single = run_multiimage_inference([img])  # 1 imagen
    pts = result_single["points"]
    individual_points.append(pts)
    az  = entry["azimuth"]
    el  = entry["elevation"]
    rot = entry["rotation_deg"]
    print(f"  [{i:02d}] AZ{az} EL{el} R{rot}  →  {len(pts)} punto(s)")

multi_result = results.get(N_VIEWS_COMPARE)
if multi_result:
    print(f"\nModo individual total: {sum(len(p) for p in individual_points)} punto(s) en {len(images_cmp)} llamadas")
    print(f"Modo multi-imagen:     {len(multi_result['points'])} punto(s) en 1 llamada")

---
## Notas sobre el formato de coordenadas

Molmo2 produce coordenadas en el rango **0–1000** con el **origen en la esquina inferior-izquierda** (Y invertido respecto a la convención de imágenes).

Para convertir a píxeles:
```python
px = point["x"] / 1000 * image_width
py = image_height - point["y"] / 1000 * image_height   # invertir Y
```

En modo multi-imagen, el campo `obj_id` indica a qué imagen del conjunto pertenece cada punto (1-based). Si el modelo devuelve `obj_id=0`, el punto no está asociado a una imagen específica.